# RRGライフサイクル別・Barra残差リターン評価 — Narrative Beta方式

このノートブックは、RRGで定義したテーマ・ライフサイクル状態ごとに、将来のBarra残差リターンを評価します。
前回のRRGライフサイクル評価ノートブックとの違いは、**Narrative Signalを外部ファイルから読むのではなく、daily narrative intensityとテーマ残差リターンからrolling narrative betaを推定し、`NarrativeSignal = Σ beta × narrative shock`として構築する**点です。

中心仮説は次です。

\[
E\left[R^{resid}_{g,t+1:t+h}\mid RRGState_{g,t}, APWCZ_{g,t}, NarrativeSignal_{g,t}\right]
\]

特に重要なのは、

\[
S_{g,t}=Improving,\quad APWCZ_{g,t}>0,\quad NarrativeSignal_{g,t}>0
\]

のテーマが、その後の残差リターン、RRG Leading入り、Leading持続性で優位かどうかです。

## 入力データ

実データで使う場合は、`CONFIG.DATA_DIR`に以下のCSVを置いてください。

必須:

- `daily_narrative_intensity.csv`
  - long形式: `date, narrative_id, I_negative` など
  - wide形式: `date, AI, Inflation, ...` でも可
- `theme_residual_returns.csv`
  - `date, theme_id, residual_return`
- `theme_apwc.csv`
  - `date, theme_id, APWC` または `date, theme_id, APWC_z`

任意:

- `theme_rrg.csv`
  - `date, theme_id, RSRatio, RSMomentum, RRG_state`

`theme_rrg.csv`がない場合は、テーマ残差リターンからRRG指標を計算します。入力ファイルがない場合はsynthetic demo dataを自動生成して全セルが動くようにしています。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

try:
    from sklearn.mixture import GaussianMixture
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    HAS_SKLEARN = True
except Exception:
    HAS_SKLEARN = False

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

## 1. 設定

主な設定はここで変更します。

- `INTENSITY_COL`: narrative intensityとして使う列。まずは`I_negative`を推奨。
- `SHOCK_WINDOWS`: daily intensityから作るattention shockの窓。20D, 60D, 120Dを初期値にしています。
- `BETA_SIGNAL_WINDOW`: narrative beta推定に使うshock列の窓。
- `BETA_WINDOW`: rolling beta推定窓。
- `RIDGE_ALPHA`: narrative数が多い場合のridge正則化。
- `BETA_REFIT_EVERY`: rolling betaを何営業日ごとに再推定するか。

In [ ]:
@dataclass
class CONFIG:
    DATA_DIR: str = "/mnt/data/rrg_beta_lifecycle_input"
    OUTPUT_DIR: str = "/mnt/data/rrg_beta_lifecycle_outputs"

    # Input files
    DAILY_NARRATIVE_INTENSITY_FILE: str = "daily_narrative_intensity.csv"
    THEME_RESIDUAL_RETURNS_FILE: str = "theme_residual_returns.csv"
    THEME_APWC_FILE: str = "theme_apwc.csv"
    THEME_RRG_FILE: str = "theme_rrg.csv"  # optional

    # Narrative intensity and shock settings
    INTENSITY_COL: str = "I_negative"  # fallback to I_total if unavailable
    SHOCK_WINDOWS: tuple = (20, 60, 120)
    SHOCK_Z_WINDOW: int = 252
    SHOCK_Z_MIN_PERIODS: int = 126
    BETA_SIGNAL_WINDOW: int = 60  # use ZShock_60D for beta-weighted signal by default

    # Rolling narrative beta settings
    BETA_WINDOW: int = 252
    BETA_MIN_OBS: int = 126
    BETA_REFIT_EVERY: int = 20
    RIDGE_ALPHA: float = 10.0
    TOP_K_BETAS: int | None = None  # if not None, use only top abs-beta narratives for signal

    # RRG settings
    RRG_RS_WINDOW: int = 120
    RRG_MOM_LAG: int = 20
    RRG_MOM_Z_WINDOW: int = 120
    RRG_MIN_PERIODS: int = 60
    RRG_HYSTERESIS_BAND: float = 0.0  # set e.g. 0.1 to reduce boundary noise

    # APWC / momentum settings
    APWC_Z_WINDOW: int = 252
    APWC_Z_MIN_PERIODS: int = 126
    RMOM_WINDOWS: tuple = (20, 60, 120)
    VOL_WINDOW: int = 60

    # Forward return horizons
    HORIZONS: tuple = (5, 20, 60, 120)

    # Flags
    SAVE_OUTPUTS: bool = False
    RANDOM_SEED: int = 42

Path(CONFIG.DATA_DIR).mkdir(parents=True, exist_ok=True)
Path(CONFIG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
CONFIG

## 2. ユーティリティ関数

このセルでは、rolling z-score、将来累積リターン、RRG状態、回帰補助関数などを定義します。

In [ ]:
def rolling_zscore(s: pd.Series, window: int, min_periods: int | None = None) -> pd.Series:
    if min_periods is None:
        min_periods = max(20, window // 2)
    m = s.rolling(window, min_periods=min_periods).mean()
    sd = s.rolling(window, min_periods=min_periods).std()
    return (s - m) / sd.replace(0, np.nan)


def future_sum(s: pd.Series, horizon: int) -> pd.Series:
    # At time t, sum s[t+1:t+horizon].
    arr = s.to_numpy(dtype=float)
    out = np.full(len(arr), np.nan)
    for i in range(len(arr) - horizon):
        w = arr[i+1:i+1+horizon]
        if np.isfinite(w).all():
            out[i] = w.sum()
    return pd.Series(out, index=s.index)


def max_drawdown(x: pd.Series) -> float:
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    nav = (1 + x).cumprod()
    dd = nav / nav.cummax() - 1
    return float(dd.min())


def ann_stats(daily_ret: pd.Series, periods_per_year: int = 252) -> dict:
    r = pd.Series(daily_ret).dropna()
    if len(r) < 3:
        return dict(mean=np.nan, vol=np.nan, ir=np.nan, hit_rate=np.nan, max_dd=np.nan, n=len(r))
    mean = r.mean() * periods_per_year
    vol = r.std(ddof=1) * np.sqrt(periods_per_year)
    return dict(
        mean=mean,
        vol=vol,
        ir=mean / vol if vol > 0 else np.nan,
        hit_rate=(r > 0).mean(),
        max_dd=max_drawdown(r),
        n=len(r)
    )


def classify_rrg_state(rs_ratio: float, rs_mom: float, band: float = 0.0) -> str:
    if pd.isna(rs_ratio) or pd.isna(rs_mom):
        return np.nan
    # simple band: values near zero are treated by sign without neutral state; set band=0 for standard RRG
    if rs_ratio > band and rs_mom > band:
        return "Leading"
    if rs_ratio > band and rs_mom <= -band:
        return "Weakening"
    if rs_ratio <= -band and rs_mom <= -band:
        return "Lagging"
    if rs_ratio <= -band and rs_mom > band:
        return "Improving"
    # within band, fall back to standard signs
    if rs_ratio >= 0 and rs_mom >= 0:
        return "Leading"
    if rs_ratio >= 0 and rs_mom < 0:
        return "Weakening"
    if rs_ratio < 0 and rs_mom < 0:
        return "Lagging"
    return "Improving"


def safe_qcut_flag_by_date(df: pd.DataFrame, value_col: str, date_col: str = "date", high_q: float = 0.5) -> pd.Series:
    # Cross-sectional high flag by date. Default: above median.
    return df.groupby(date_col)[value_col].transform(lambda x: x.rank(pct=True)) >= high_q


def fit_ols(formula: str, data: pd.DataFrame, cluster_col: str | None = None):
    if not HAS_STATSMODELS:
        print("statsmodels is unavailable.")
        return None
    d = data.replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < 50:
        print(f"Insufficient rows for regression: {len(d)}")
        return None
    try:
        model = smf.ols(formula, data=d).fit()
        if cluster_col is not None and cluster_col in d.columns:
            model = model.get_robustcov_results(cov_type="cluster", groups=d[cluster_col])
        return model
    except Exception as e:
        print("Regression failed:", e)
        return None


def print_model_summary(model, max_rows: int = 20):
    if model is None:
        return
    try:
        coefs = pd.DataFrame({
            "coef": model.params,
            "t": model.tvalues,
            "p": model.pvalues,
        })
        display(coefs.head(max_rows))
        print(f"N={int(model.nobs)}, R2={getattr(model, 'rsquared', np.nan):.4f}")
    except Exception:
        print(model.summary())

## 3. データ読込またはsynthetic demo data生成

実データがない場合は、分析パイプライン検証用のsynthetic dataを生成します。
synthetic dataでは、いくつかのテーマが特定narrativeに正のbetaを持ち、Narrative SignalがAPWCや残差リターン、RRG状態に影響するように設計しています。

In [ ]:
def generate_synthetic_data(seed: int = CONFIG.RANDOM_SEED):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2019-01-02", "2024-12-31")
    themes = [f"Theme_{i:02d}" for i in range(1, 13)]
    narratives = [f"Narrative_{j:02d}" for j in range(1, 11)]

    # Narrative intensity: persistent positive processes with occasional waves
    rows = []
    for n_idx, n in enumerate(narratives):
        x = np.zeros(len(dates))
        shocks = rng.normal(0, 0.01, len(dates))
        for t in range(1, len(dates)):
            x[t] = 0.96 * x[t-1] + shocks[t]
        # add waves
        for _ in range(5):
            start = rng.integers(100, len(dates)-150)
            length = rng.integers(40, 160)
            amp = rng.uniform(0.005, 0.03)
            wave = amp * np.exp(-np.arange(length) / rng.uniform(40, 120))
            x[start:start+length] += wave[:max(0, min(length, len(dates)-start))]
        intensity = np.clip(0.015 + x + rng.normal(0, 0.002, len(dates)), 0, None)
        for d, val in zip(dates, intensity):
            rows.append({"date": d, "narrative_id": n, "I_negative": val, "I_total": val * rng.uniform(1.5, 2.5)})
    intensity = pd.DataFrame(rows)

    # Build preliminary shocks for return generation
    piv = intensity.pivot(index="date", columns="narrative_id", values="I_negative").sort_index().fillna(0.0)
    shock = piv.rolling(60, min_periods=20).mean().shift(1) - piv.rolling(60, min_periods=20).mean().shift(61)
    shock = shock.apply(lambda s: rolling_zscore(s, 252, 80)).fillna(0.0)

    # True theme-narrative betas, sparse
    true_beta = pd.DataFrame(0.0, index=themes, columns=narratives)
    for theme in themes:
        chosen = rng.choice(narratives, size=3, replace=False)
        true_beta.loc[theme, chosen] = rng.normal(0.0015, 0.0008, size=3)
        # allow one negative exposure sometimes
        if rng.random() < 0.35:
            true_beta.loc[theme, rng.choice(narratives)] = rng.normal(-0.0012, 0.0005)

    residual_rows = []
    apwc_rows = []
    for theme in themes:
        beta_vec = true_beta.loc[theme].values
        signal = shock[narratives].values @ beta_vec
        # residual returns with some persistence and narrative effects
        eps = np.zeros(len(dates))
        noise = rng.normal(0, 0.010, len(dates))
        for t in range(1, len(dates)):
            eps[t] = 0.07 * eps[t-1] + signal[t] + noise[t]
        # APWC related to abs signal and momentum with noise
        rmom60 = pd.Series(eps, index=dates).rolling(60, min_periods=20).sum().shift(1).fillna(0.0).to_numpy()
        apwc_raw = 0.10 + 8.0 * np.abs(signal) + 0.5 * np.maximum(rmom60, 0) + rng.normal(0, 0.04, len(dates))
        apwc_raw = np.clip(apwc_raw, -0.10, 0.80)
        for d, r, a in zip(dates, eps, apwc_raw):
            residual_rows.append({"date": d, "theme_id": theme, "residual_return": r})
            apwc_rows.append({"date": d, "theme_id": theme, "APWC": a})

    residual = pd.DataFrame(residual_rows)
    apwc = pd.DataFrame(apwc_rows)
    return intensity, residual, apwc, None


def load_or_generate_data():
    data_dir = Path(CONFIG.DATA_DIR)
    int_path = data_dir / CONFIG.DAILY_NARRATIVE_INTENSITY_FILE
    ret_path = data_dir / CONFIG.THEME_RESIDUAL_RETURNS_FILE
    apwc_path = data_dir / CONFIG.THEME_APWC_FILE
    rrg_path = data_dir / CONFIG.THEME_RRG_FILE

    if int_path.exists() and ret_path.exists() and apwc_path.exists():
        print("Loading real data from", data_dir)
        intensity = pd.read_csv(int_path)
        residual = pd.read_csv(ret_path)
        apwc = pd.read_csv(apwc_path)
        rrg = pd.read_csv(rrg_path) if rrg_path.exists() else None
    else:
        print("Input files not found. Generating synthetic demo data.")
        intensity, residual, apwc, rrg = generate_synthetic_data()

    for df in [intensity, residual, apwc] + ([] if rrg is None else [rrg]):
        if df is not None and "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"])
    return intensity, residual, apwc, rrg

intensity_raw, residual_raw, apwc_raw, rrg_raw = load_or_generate_data()
print(intensity_raw.head())
print(residual_raw.head())
print(apwc_raw.head())

## 4. Daily narrative intensityの整形

`daily_narrative_intensity.csv`はlong形式・wide形式の両方に対応します。

- long形式: `date, narrative_id, I_negative`
- wide形式: `date, AI, Inflation, ...`

このセルでは、long形式に揃え、分析対象のintensity列を確定します。

In [ ]:
def normalize_intensity_format(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    # long format if narrative_id exists
    if "narrative_id" in df.columns:
        out = df.copy()
        if CONFIG.INTENSITY_COL not in out.columns:
            fallback_cols = [c for c in ["I_total", "I_negative", "intensity", "value"] if c in out.columns]
            if not fallback_cols:
                raise ValueError("No usable intensity column found.")
            print(f"CONFIG.INTENSITY_COL={CONFIG.INTENSITY_COL} not found. Using {fallback_cols[0]}")
            out[CONFIG.INTENSITY_COL] = out[fallback_cols[0]]
        return out[["date", "narrative_id", CONFIG.INTENSITY_COL] + [c for c in out.columns if c.startswith("I_") and c != CONFIG.INTENSITY_COL]].copy()

    # wide format: date + narrative columns
    value_cols = [c for c in df.columns if c != "date"]
    out = df.melt(id_vars="date", value_vars=value_cols, var_name="narrative_id", value_name=CONFIG.INTENSITY_COL)
    return out

intensity = normalize_intensity_format(intensity_raw)
intensity = intensity.sort_values(["narrative_id", "date"]).reset_index(drop=True)
print(intensity.head())
print("Narratives:", intensity["narrative_id"].nunique(), "Date range:", intensity["date"].min(), intensity["date"].max())

## 5. Narrative shock / smoothness / persistenceの作成

Daily intensityから、Narrative Momentum型のattention shockを作ります。

\[
Shock^J_{n,t}=\overline{I}_{n,t-J:t-1}-\overline{I}_{n,t-2J:t-J-1}
\]

また、rolling z-score、smoothness、AR(1)的persistenceも作ります。

In [ ]:
def add_narrative_features(intensity: pd.DataFrame) -> pd.DataFrame:
    df = intensity.sort_values(["narrative_id", "date"]).copy()
    col = CONFIG.INTENSITY_COL
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
    df["dI"] = df.groupby("narrative_id")[col].diff()

    for w in CONFIG.SHOCK_WINDOWS:
        recent = df.groupby("narrative_id")[col].transform(lambda x: x.shift(1).rolling(w, min_periods=max(5, w//2)).mean())
        prev = df.groupby("narrative_id")[col].transform(lambda x: x.shift(1+w).rolling(w, min_periods=max(5, w//2)).mean())
        df[f"Shock_{w}D"] = recent - prev
        df[f"ZShock_{w}D"] = df.groupby("narrative_id")[f"Shock_{w}D"].transform(
            lambda x: rolling_zscore(x, CONFIG.SHOCK_Z_WINDOW, CONFIG.SHOCK_Z_MIN_PERIODS)
        )
        mean_d = df.groupby("narrative_id")["dI"].transform(lambda x: x.shift(1).rolling(w, min_periods=max(5, w//2)).mean())
        std_d = df.groupby("narrative_id")["dI"].transform(lambda x: x.shift(1).rolling(w, min_periods=max(5, w//2)).std())
        df[f"Smoothness_{w}D"] = mean_d.abs() / std_d.replace(0, np.nan)

    # rolling AR(1) persistence approximation: corr(I_t, I_{t-1}) over 252 days
    df["I_lag1"] = df.groupby("narrative_id")[col].shift(1)
    df["Persistence_252D"] = df.groupby("narrative_id", group_keys=False).apply(
        lambda g: g[col].rolling(252, min_periods=126).corr(g["I_lag1"])
    ).reset_index(level=0, drop=True)
    return df

narr_feat = add_narrative_features(intensity)
print(narr_feat.head())

## 6. テーマ残差リターン、APWC、RRGの準備

このノートブックでは、RRG状態別の**将来Barra残差リターン**を評価します。
APWCがrawで与えられている場合はrolling z-scoreを作ります。RRGが与えられていない場合は、テーマ残差リターンからRRG座標と状態を作ります。

In [ ]:
def prepare_residual_returns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"])
    if "theme_id" not in out.columns:
        raise ValueError("theme_residual_returns must contain theme_id")
    ret_col = None
    for c in ["residual_return", "bottom_up_residual_return", "ret", "return"]:
        if c in out.columns:
            ret_col = c
            break
    if ret_col is None:
        raise ValueError("No residual return column found.")
    out = out.rename(columns={ret_col: "residual_return"})
    out["residual_return"] = pd.to_numeric(out["residual_return"], errors="coerce")
    return out[["date", "theme_id", "residual_return"]].sort_values(["theme_id", "date"])


def prepare_apwc(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"])
    if "APWC_z" not in out.columns:
        if "APWC" not in out.columns:
            raise ValueError("theme_apwc must contain APWC or APWC_z")
        out["APWC"] = pd.to_numeric(out["APWC"], errors="coerce")
        out["APWC_z"] = out.groupby("theme_id")["APWC"].transform(
            lambda x: rolling_zscore(x, CONFIG.APWC_Z_WINDOW, CONFIG.APWC_Z_MIN_PERIODS)
        )
    else:
        out["APWC_z"] = pd.to_numeric(out["APWC_z"], errors="coerce")
        if "APWC" not in out.columns:
            out["APWC"] = np.nan
    return out[["date", "theme_id", "APWC", "APWC_z"]].sort_values(["theme_id", "date"])

residual = prepare_residual_returns(residual_raw)
apwc = prepare_apwc(apwc_raw)
print(residual.head())
print(apwc.head())

In [ ]:
def compute_rrg_from_residual(residual: pd.DataFrame) -> pd.DataFrame:
    df = residual.sort_values(["theme_id", "date"]).copy()
    # residual log NAV by theme
    df["theme_log_nav"] = df.groupby("theme_id")["residual_return"].cumsum()
    # benchmark: equal-weight residual return across themes per day
    bench_ret = df.groupby("date")["residual_return"].mean().rename("bench_resid_ret").reset_index()
    bench_ret["bench_log_nav"] = bench_ret["bench_resid_ret"].cumsum()
    df = df.merge(bench_ret[["date", "bench_log_nav"]], on="date", how="left")
    df["RS"] = df["theme_log_nav"] - df["bench_log_nav"]
    df["RSRatio"] = df.groupby("theme_id")["RS"].transform(
        lambda x: rolling_zscore(x, CONFIG.RRG_RS_WINDOW, CONFIG.RRG_MIN_PERIODS)
    )
    raw_mom = df.groupby("theme_id")["RSRatio"].diff(CONFIG.RRG_MOM_LAG)
    df["RSMomentum"] = raw_mom.groupby(df["theme_id"]).transform(
        lambda x: rolling_zscore(x, CONFIG.RRG_MOM_Z_WINDOW, CONFIG.RRG_MIN_PERIODS)
    )
    df["RRG_state"] = [classify_rrg_state(a, b, CONFIG.RRG_HYSTERESIS_BAND) for a, b in zip(df["RSRatio"], df["RSMomentum"])]
    return df[["date", "theme_id", "RS", "RSRatio", "RSMomentum", "RRG_state"]]


def prepare_rrg(rrg_raw, residual):
    if rrg_raw is not None:
        out = rrg_raw.copy()
        out["date"] = pd.to_datetime(out["date"])
        if "RRG_state" not in out.columns:
            state_col = next((c for c in ["state", "rrg_state"] if c in out.columns), None)
            if state_col:
                out = out.rename(columns={state_col: "RRG_state"})
        if "RSRatio" not in out.columns or "RSMomentum" not in out.columns or "RRG_state" not in out.columns:
            print("theme_rrg exists but missing columns. Recomputing RRG from residual returns.")
            return compute_rrg_from_residual(residual)
        return out[["date", "theme_id", "RSRatio", "RSMomentum", "RRG_state"]]
    return compute_rrg_from_residual(residual)

rrg = prepare_rrg(rrg_raw, residual)
print(rrg.head())
print(rrg["RRG_state"].value_counts(dropna=False))

## 7. Residual Momentum、残差ボラティリティ、将来残差リターン

RRG状態ごとの将来残差リターンを評価するため、各horizonのforward residual returnを作ります。

In [ ]:
def add_theme_features(residual: pd.DataFrame) -> pd.DataFrame:
    df = residual.sort_values(["theme_id", "date"]).copy()
    for w in CONFIG.RMOM_WINDOWS:
        df[f"RMOM_{w}D"] = df.groupby("theme_id")["residual_return"].transform(
            lambda x: x.shift(1).rolling(w, min_periods=max(5, w//2)).sum()
        )
    df[f"Vol_{CONFIG.VOL_WINDOW}D"] = df.groupby("theme_id")["residual_return"].transform(
        lambda x: x.shift(1).rolling(CONFIG.VOL_WINDOW, min_periods=max(10, CONFIG.VOL_WINDOW//2)).std()
    )
    for h in CONFIG.HORIZONS:
        df[f"FwdRet_{h}D"] = df.groupby("theme_id")["residual_return"].transform(lambda x: future_sum(x, h))
    return df

theme_feat = add_theme_features(residual)
base = theme_feat.merge(apwc, on=["date", "theme_id"], how="left").merge(rrg, on=["date", "theme_id"], how="left")
print(base.head())

## 8. Rolling narrative beta推定

ここからがNarrative Beta方式の中核です。

1. 日次narrative intensityから`ZShock_{BETA_SIGNAL_WINDOW}D`を作る。
2. 各テーマのBarra残差リターンを、そのnarrative shock群にrolling ridge回帰する。
3. 推定したテーマ別narrative betaと当日のnarrative shockを掛け合わせ、`NarrativeSignal`を作る。

\[
\epsilon_{g,t}=\alpha_g+\Gamma_{g,t}^{\top}ZShock_t+u_{g,t}
\]

\[
NarrativeSignal_{g,t}=\sum_n \gamma_{g,n,t}\,ZShock_{n,t}
\]

In [ ]:
def make_shock_wide(narr_feat: pd.DataFrame, window: int) -> pd.DataFrame:
    col = f"ZShock_{window}D"
    if col not in narr_feat.columns:
        raise ValueError(f"{col} not found in narrative features.")
    wide = narr_feat.pivot(index="date", columns="narrative_id", values=col).sort_index()
    wide = wide.replace([np.inf, -np.inf], np.nan)
    return wide

shock_wide = make_shock_wide(narr_feat, CONFIG.BETA_SIGNAL_WINDOW)
print(shock_wide.head())
print("Shock wide shape:", shock_wide.shape)

In [ ]:
def ridge_beta(X: np.ndarray, y: np.ndarray, alpha: float) -> np.ndarray:
    # X and y should be demeaned; no intercept estimated here.
    p = X.shape[1]
    XtX = X.T @ X
    Xty = X.T @ y
    return np.linalg.solve(XtX + alpha * np.eye(p), Xty)


def estimate_theme_narrative_betas(residual: pd.DataFrame, shock_wide: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Estimate rolling ridge betas and create beta-weighted NarrativeSignal.
    # Returns signal_df and beta_df.
    ret_wide = residual.pivot(index="date", columns="theme_id", values="residual_return").sort_index()
    dates = ret_wide.index.intersection(shock_wide.index)
    X_all = shock_wide.loc[dates].astype(float)
    ret_wide = ret_wide.loc[dates].astype(float)
    narratives = list(X_all.columns)
    themes = list(ret_wide.columns)

    signal_records = []
    beta_records = []

    X_values = X_all.to_numpy(dtype=float)
    for theme in themes:
        y_values = ret_wide[theme].to_numpy(dtype=float)
        last_beta = np.full(len(narratives), np.nan)
        for i, d in enumerate(dates):
            refit = (i >= CONFIG.BETA_WINDOW) and ((i - CONFIG.BETA_WINDOW) % CONFIG.BETA_REFIT_EVERY == 0)
            if refit:
                start = i - CONFIG.BETA_WINDOW
                Xw = X_values[start:i, :]
                yw = y_values[start:i]
                mask = np.isfinite(yw) & np.all(np.isfinite(Xw), axis=1)
                if mask.sum() >= CONFIG.BETA_MIN_OBS:
                    X = Xw[mask]
                    y = yw[mask]
                    # de-mean for intercept; standardize X within window for stability
                    X_mean = X.mean(axis=0)
                    X_std = X.std(axis=0)
                    X_std[X_std == 0] = 1.0
                    Xs = (X - X_mean) / X_std
                    ys = y - y.mean()
                    try:
                        b_std = ridge_beta(Xs, ys, CONFIG.RIDGE_ALPHA)
                        last_beta = b_std / X_std
                    except np.linalg.LinAlgError:
                        last_beta = np.full(len(narratives), np.nan)

            x_today = X_values[i, :]
            beta = last_beta.copy()
            if CONFIG.TOP_K_BETAS is not None and np.isfinite(beta).any():
                keep = np.argsort(np.abs(beta))[-CONFIG.TOP_K_BETAS:]
                beta_masked = np.zeros_like(beta)
                beta_masked[keep] = beta[keep]
                beta = beta_masked

            if np.isfinite(beta).any() and np.all(np.isfinite(x_today)):
                signal = float(np.nansum(beta * x_today))
                beta_norm = float(np.nansum(np.abs(beta)))
                if beta_norm > 0:
                    attention_abs = float(np.nansum(np.abs(beta) * np.maximum(x_today, 0)) / beta_norm)
                else:
                    attention_abs = np.nan
                top_idx = np.argsort(np.abs(beta))[-3:][::-1]
                top_names = ";".join([f"{narratives[j]}:{beta[j]:.4g}" for j in top_idx if np.isfinite(beta[j])])
            else:
                signal = np.nan
                beta_norm = np.nan
                attention_abs = np.nan
                top_names = ""

            signal_records.append({
                "date": d,
                "theme_id": theme,
                "NarrativeSignal": signal,
                "NarrativeAttentionAbs": attention_abs,
                "BetaNorm": beta_norm,
                "TopNarratives": top_names,
            })

            # Store betas at refit dates only to reduce output size
            if refit and np.isfinite(last_beta).any():
                for n, b in zip(narratives, last_beta):
                    beta_records.append({"date": d, "theme_id": theme, "narrative_id": n, "beta": b})

    signal_df = pd.DataFrame(signal_records)
    beta_df = pd.DataFrame(beta_records)
    return signal_df, beta_df

narr_signal, beta_panel = estimate_theme_narrative_betas(residual, shock_wide)
print(narr_signal.head())
print(beta_panel.head())
print("Signals:", narr_signal.shape, "Betas:", beta_panel.shape)

## 9. 分析用パネルの作成

ここで、残差リターン、APWC、RRG、Narrative Beta方式のNarrative Signalを結合します。
また、APWC High / Narrative Highなどの分類フラグを作ります。

In [ ]:
panel = base.merge(narr_signal, on=["date", "theme_id"], how="left")

# Cross-sectional flags by date
panel["APWC_high"] = safe_qcut_flag_by_date(panel, "APWC_z", high_q=0.5)
panel["Narrative_high"] = panel["NarrativeSignal"] > 0
panel["Narrative_rank_high"] = safe_qcut_flag_by_date(panel, "NarrativeSignal", high_q=0.5)

# State dummies
for s in ["Leading", "Weakening", "Lagging", "Improving"]:
    panel[f"State_{s}"] = (panel["RRG_state"] == s).astype(int)

# Combined lifecycle labels
panel["RRG_APWC_Narrative_Group"] = (
    panel["RRG_state"].astype(str)
    + " | APWC_" + np.where(panel["APWC_high"], "High", "Low")
    + " | Narrative_" + np.where(panel["Narrative_high"], "High", "Low")
)

print(panel.head())
print(panel[["RRG_state", "APWC_high", "Narrative_high"]].value_counts(dropna=False).head(20))

## 10. RRG状態別の将来残差リターン

まず、RRG状態ごとの将来残差リターンを評価します。
ここでの将来残差リターンは、状態観測日の翌営業日からhorizon終了日までの累積Barra残差リターンです。

In [ ]:
def summarize_forward_by_group(df: pd.DataFrame, group_cols: list[str], horizon: int) -> pd.DataFrame:
    col = f"FwdRet_{horizon}D"
    d = df[group_cols + [col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) == 0:
        return pd.DataFrame()
    out = d.groupby(group_cols)[col].agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
        hit_rate=lambda x: (x > 0).mean(),
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
    ).reset_index()
    out["t_stat_naive"] = out["mean"] / (out["std"] / np.sqrt(out["n"]))
    out["ir_per_horizon"] = out["mean"] / out["std"]
    out["horizon"] = horizon
    return out

rrg_state_summaries = []
for h in CONFIG.HORIZONS:
    rrg_state_summaries.append(summarize_forward_by_group(panel, ["RRG_state"], h))
rrg_state_summary = pd.concat(rrg_state_summaries, ignore_index=True)
display(rrg_state_summary)

In [ ]:
# Plot state-wise forward returns for a selected horizon
h = 20
plot_df = rrg_state_summary[rrg_state_summary["horizon"] == h].dropna(subset=["RRG_state"]).copy()
order = ["Improving", "Leading", "Weakening", "Lagging"]
plot_df["RRG_state"] = pd.Categorical(plot_df["RRG_state"], categories=order, ordered=True)
plot_df = plot_df.sort_values("RRG_state")
plt.figure(figsize=(8, 4))
plt.bar(plot_df["RRG_state"].astype(str), plot_df["mean"])
plt.axhline(0, linewidth=1)
plt.title(f"Mean forward residual return by RRG state ({h}D)")
plt.ylabel("Forward residual return")
plt.show()

## 11. RRG状態ダミーのパネル回帰

Laggingを基準状態として、Leading / Weakening / Improvingが将来残差リターンを説明するかを確認します。
テーマ固定効果と月固定効果を入れます。

In [ ]:
# Month fixed effects instead of daily fixed effects to avoid excessive dummy dimensions
panel["month"] = panel["date"].dt.to_period("M").astype(str)

h = 20
reg_df = panel.copy()
reg_df["y"] = reg_df[f"FwdRet_{h}D"]
formula = "y ~ State_Leading + State_Weakening + State_Improving + APWC_z + NarrativeSignal + RMOM_60D + Vol_60D + C(theme_id) + C(month)"
model_rrg = fit_ols(formula, reg_df, cluster_col="date")
print_model_summary(model_rrg, max_rows=15)

## 12. RRG状態 × APWC High/Low

同じRRG状態でも、APWCが高いか低いかで将来残差リターンが異なるかを確認します。
APWCが高いLeadingやImprovingは、単なる価格モメンタムではなく、テーマ内銘柄の残差共振を伴う状態と解釈できます。

In [ ]:
rrg_apwc_summaries = []
for h in CONFIG.HORIZONS:
    rrg_apwc_summaries.append(summarize_forward_by_group(panel, ["RRG_state", "APWC_high"], h))
rrg_apwc_summary = pd.concat(rrg_apwc_summaries, ignore_index=True)
display(rrg_apwc_summary.head(30))

h = 20
pivot = rrg_apwc_summary[rrg_apwc_summary["horizon"] == h].pivot(index="RRG_state", columns="APWC_high", values="mean")
display(pivot)

## 13. RRG状態 × Narrative Signal High/Low

ここでは、RRG状態ごとにNarrative Signalが正か負かで将来残差リターンが異なるかを確認します。
このNarrative Signalはrolling narrative betaで構築されています。

In [ ]:
rrg_narr_summaries = []
for h in CONFIG.HORIZONS:
    rrg_narr_summaries.append(summarize_forward_by_group(panel, ["RRG_state", "Narrative_high"], h))
rrg_narr_summary = pd.concat(rrg_narr_summaries, ignore_index=True)
display(rrg_narr_summary.head(30))

h = 20
pivot = rrg_narr_summary[rrg_narr_summary["horizon"] == h].pivot(index="RRG_state", columns="Narrative_high", values="mean")
display(pivot)

## 14. RRG状態 × APWC × Narrative Signal

実務的に最も重要なクロス分類です。

特に見たいのは、

\[
S_{g,t}=Improving,\quad APWC\ High,\quad Narrative\ High
\]

のテーマが、その後の残差リターンで優位かどうかです。

In [ ]:
rrg_apwc_narr_summaries = []
for h in CONFIG.HORIZONS:
    rrg_apwc_narr_summaries.append(summarize_forward_by_group(panel, ["RRG_state", "APWC_high", "Narrative_high"], h))
rrg_apwc_narr_summary = pd.concat(rrg_apwc_narr_summaries, ignore_index=True)
display(rrg_apwc_narr_summary.head(50))

h = 20
focus = rrg_apwc_narr_summary[(rrg_apwc_narr_summary["horizon"] == h) & (rrg_apwc_narr_summary["RRG_state"].isin(["Improving", "Leading"]))]
display(focus.sort_values(["RRG_state", "APWC_high", "Narrative_high"]))

## 15. Improving × APWC × Narrative の相互作用回帰

中心仮説は、**Improving状態にあり、APWCが高く、Narrative Signalも高いテーマが、その後高い残差リターンを持つ**というものです。

ここでは、連続変数の相互作用として、`State_Improving × APWC_z × NarrativeSignal`を検証します。

In [ ]:
h = 20
reg_df = panel.copy()
reg_df["y"] = reg_df[f"FwdRet_{h}D"]
reg_df["Improving_x_APWC"] = reg_df["State_Improving"] * reg_df["APWC_z"]
reg_df["Improving_x_Narrative"] = reg_df["State_Improving"] * reg_df["NarrativeSignal"]
reg_df["APWC_x_Narrative"] = reg_df["APWC_z"] * reg_df["NarrativeSignal"]
reg_df["Improving_x_APWC_x_Narrative"] = reg_df["State_Improving"] * reg_df["APWC_z"] * reg_df["NarrativeSignal"]

formula = (
    "y ~ State_Leading + State_Weakening + State_Improving + "
    "APWC_z + NarrativeSignal + RMOM_60D + "
    "Improving_x_APWC + Improving_x_Narrative + APWC_x_Narrative + Improving_x_APWC_x_Narrative + "
    "RSRatio + RSMomentum + Vol_60D + C(theme_id) + C(month)"
)
model_interaction = fit_ols(formula, reg_df, cluster_col="date")
print_model_summary(model_interaction, max_rows=25)

## 16. RRG状態遷移ごとの将来残差リターン

RRG状態そのものではなく、状態遷移ごとに将来残差リターンを評価します。
特に、`Improving → Leading`が高い将来リターンを持つかを確認します。

In [ ]:
def add_future_state(panel: pd.DataFrame, horizon: int) -> pd.DataFrame:
    df = panel.sort_values(["theme_id", "date"]).copy()
    df[f"RRG_state_fwd_{horizon}D"] = df.groupby("theme_id")["RRG_state"].shift(-horizon)
    df[f"Transition_{horizon}D"] = df["RRG_state"].astype(str) + " -> " + df[f"RRG_state_fwd_{horizon}D"].astype(str)
    return df

transition_summaries = []
transition_panels = {}
for h in CONFIG.HORIZONS:
    p_h = add_future_state(panel, h)
    transition_panels[h] = p_h
    transition_summaries.append(summarize_forward_by_group(p_h, [f"Transition_{h}D"], h))
transition_summary = pd.concat(transition_summaries, ignore_index=True)
# show frequent transitions only
h = 20
display(transition_summary[(transition_summary["horizon"] == h) & (transition_summary["n"] > 50)].sort_values("mean", ascending=False).head(30))

## 17. RRG状態episode分析

日次観測の状態別平均は、長いLeading期間を何度も数えてしまいます。
そこで、連続した同一状態を1つのepisodeとして扱い、episode単位の期間・残差リターンを評価します。

In [ ]:
def build_state_episodes(panel: pd.DataFrame) -> pd.DataFrame:
    records = []
    df = panel.sort_values(["theme_id", "date"]).copy()
    for theme, g in df.groupby("theme_id"):
        g = g.reset_index(drop=True)
        state = g["RRG_state"].astype(object)
        # episode id increments when state changes
        ep_id = (state != state.shift(1)).cumsum()
        g["episode_id"] = ep_id
        for eid, e in g.groupby("episode_id"):
            st = e["RRG_state"].iloc[0]
            if pd.isna(st):
                continue
            ret = (1 + e["residual_return"].dropna()).prod() - 1 if e["residual_return"].notna().any() else np.nan
            records.append({
                "theme_id": theme,
                "episode_id": int(eid),
                "RRG_state": st,
                "start_date": e["date"].iloc[0],
                "end_date": e["date"].iloc[-1],
                "duration": len(e),
                "episode_return": ret,
                "mean_APWC_z": e["APWC_z"].mean(),
                "mean_NarrativeSignal": e["NarrativeSignal"].mean(),
                "start_APWC_z": e["APWC_z"].iloc[0],
                "start_NarrativeSignal": e["NarrativeSignal"].iloc[0],
            })
    return pd.DataFrame(records)

episodes = build_state_episodes(panel)
display(episodes.head())
episode_summary = episodes.groupby("RRG_state").agg(
    n=("episode_id", "count"),
    mean_duration=("duration", "mean"),
    median_duration=("duration", "median"),
    mean_episode_return=("episode_return", "mean"),
    hit_rate=("episode_return", lambda x: (x > 0).mean()),
    mean_start_APWC_z=("start_APWC_z", "mean"),
    mean_start_NarrativeSignal=("start_NarrativeSignal", "mean"),
).reset_index()
display(episode_summary)

## 18. Leading状態の年齢効果

Leadingの初期・中期・末期で将来残差リターンが異なるかを調べます。
Late Leadingで将来リターンが低下するなら、RRG Weakening前の利食いシグナルとして使える可能性があります。

In [ ]:
def add_state_age(panel: pd.DataFrame) -> pd.DataFrame:
    df = panel.sort_values(["theme_id", "date"]).copy()
    ages = []
    for theme, g in df.groupby("theme_id"):
        state = g["RRG_state"].values
        age = np.zeros(len(g), dtype=float)
        current_age = 0
        prev = None
        for i, st in enumerate(state):
            if pd.isna(st):
                current_age = 0
                age[i] = np.nan
            elif st == prev:
                current_age += 1
                age[i] = current_age
            else:
                current_age = 1
                age[i] = current_age
            prev = st
        ages.append(pd.Series(age, index=g.index))
    df["state_age"] = pd.concat(ages).sort_index()
    df["Leading_age_bucket"] = np.where(
        df["RRG_state"] != "Leading", np.nan,
        np.where(df["state_age"] <= 20, "EarlyLeading",
                 np.where(df["state_age"] <= 60, "MatureLeading", "LateLeading"))
    )
    return df

panel_age = add_state_age(panel)
leading_age_summaries = []
for h in CONFIG.HORIZONS:
    leading_age_summaries.append(summarize_forward_by_group(panel_age.dropna(subset=["Leading_age_bucket"]), ["Leading_age_bucket"], h))
leading_age_summary = pd.concat(leading_age_summaries, ignore_index=True)
display(leading_age_summary)

## 19. Leading Entryイベントスタディ

非Leading状態からLeadingに入った日をイベント日として、その後の累積残差リターンを評価します。
さらに、Leading入り時点でAPWC HighかつNarrative Highだったかで分けます。

In [ ]:
def identify_leading_entries(panel: pd.DataFrame) -> pd.DataFrame:
    df = panel.sort_values(["theme_id", "date"]).copy()
    df["prev_state"] = df.groupby("theme_id")["RRG_state"].shift(1)
    df["LeadingEntry"] = (df["RRG_state"].eq("Leading") & ~df["prev_state"].eq("Leading"))
    df["ConfirmedEntry"] = df["LeadingEntry"] & df["APWC_high"] & df["Narrative_high"]
    return df

panel_entry = identify_leading_entries(panel_age)
entry_events = panel_entry[panel_entry["LeadingEntry"]].copy()
print("Leading entries:", len(entry_events), "Confirmed entries:", entry_events["ConfirmedEntry"].sum())
display(entry_events[["date", "theme_id", "APWC_z", "NarrativeSignal", "ConfirmedEntry", "FwdRet_20D", "FwdRet_60D"]].head())

# Event summary by confirmed flag
entry_summary = []
for h in CONFIG.HORIZONS:
    entry_summary.append(summarize_forward_by_group(entry_events, ["ConfirmedEntry"], h))
entry_summary = pd.concat(entry_summary, ignore_index=True)
display(entry_summary)

In [ ]:
def event_study(panel: pd.DataFrame, events: pd.DataFrame, window_pre: int = 20, window_post: int = 60) -> pd.DataFrame:
    panel_idx = panel.set_index(["theme_id", "date"]).sort_index()
    records = []
    for _, ev in events.iterrows():
        theme = ev["theme_id"]
        date = ev["date"]
        g = panel[panel["theme_id"] == theme].sort_values("date").reset_index(drop=True)
        locs = np.where(g["date"].values == np.datetime64(date))[0]
        if len(locs) == 0:
            continue
        loc = locs[0]
        start = max(0, loc - window_pre)
        end = min(len(g) - 1, loc + window_post)
        sub = g.iloc[start:end+1].copy()
        sub["event_time"] = np.arange(start - loc, end - loc + 1)
        sub["event_theme"] = theme
        sub["event_date"] = date
        sub["ConfirmedEntry"] = bool(ev["ConfirmedEntry"])
        # cumulative residual return from event date forward/backward using simple cumsum around 0
        sub["cum_resid_from_event"] = sub["residual_return"].cumsum() - sub.loc[sub["event_time"] == 0, "residual_return"].iloc[0]
        records.append(sub[["event_time", "event_theme", "event_date", "ConfirmedEntry", "residual_return", "cum_resid_from_event", "APWC_z", "NarrativeSignal", "RSRatio", "RSMomentum"]])
    if not records:
        return pd.DataFrame()
    return pd.concat(records, ignore_index=True)

es = event_study(panel_entry, entry_events, 20, 60)
if len(es) > 0:
    avg_es = es.groupby(["event_time", "ConfirmedEntry"])["cum_resid_from_event"].mean().reset_index()
    plt.figure(figsize=(9, 5))
    for flag, g in avg_es.groupby("ConfirmedEntry"):
        plt.plot(g["event_time"], g["cum_resid_from_event"], label=f"ConfirmedEntry={flag}")
    plt.axvline(0, linewidth=1)
    plt.axhline(0, linewidth=1)
    plt.legend()
    plt.title("Leading Entry Event Study: cumulative residual return")
    plt.xlabel("Event time")
    plt.ylabel("Average cumulative residual return")
    plt.show()
else:
    print("No event study records.")

## 20. False Leading分析

Leading入りしたにもかかわらず、一定期間内に十分Leading滞在できなかったケースをFalse Leadingと定義します。
APWCとNarrative Signalが高いLeading入りは、False Leadingが少ないかを確認します。

In [ ]:
def add_false_leading(entry_events: pd.DataFrame, panel: pd.DataFrame, H: int = 20, K: int = 5) -> pd.DataFrame:
    panel_sorted = panel.sort_values(["theme_id", "date"]).copy()
    records = []
    for _, ev in entry_events.iterrows():
        theme = ev["theme_id"]
        date = ev["date"]
        g = panel_sorted[panel_sorted["theme_id"] == theme].sort_values("date").reset_index(drop=True)
        locs = np.where(g["date"].values == np.datetime64(date))[0]
        if len(locs) == 0:
            continue
        loc = locs[0]
        future = g.iloc[loc:loc+H]
        lead_days = future["RRG_state"].eq("Leading").sum()
        rec = ev.to_dict()
        rec["LeadDaysNextH"] = lead_days
        rec["FalseLeading"] = lead_days < K
        records.append(rec)
    return pd.DataFrame(records)

false_lead = add_false_leading(entry_events, panel_entry, H=20, K=5)
if len(false_lead) > 0:
    display(false_lead.groupby(["APWC_high", "Narrative_high"])["FalseLeading"].agg(["count", "mean"]).reset_index())
    if HAS_STATSMODELS and false_lead["FalseLeading"].nunique() > 1:
        fl = false_lead.copy()
        fl["FalseLeadingInt"] = fl["FalseLeading"].astype(int)
        try:
            logit_model = smf.logit("FalseLeadingInt ~ APWC_z + NarrativeSignal + RMOM_60D + RSRatio + RSMomentum", data=fl.replace([np.inf, -np.inf], np.nan).dropna()).fit(disp=False)
            display(pd.DataFrame({"coef": logit_model.params, "z": logit_model.tvalues, "p": logit_model.pvalues}))
        except Exception as e:
            print("False leading logit failed:", e)
else:
    print("No leading entries for false leading analysis.")

## 21. RRGライフサイクル戦略比較

RRG状態を使った単純戦略を比較します。
ここではすべてテーマ残差リターンで評価します。

- Leading Strategy
- Improving Strategy
- Improving + APWC High + Narrative High
- Confirmed Leading
- Exit-aware Strategy

In [ ]:
def make_strategy_returns(panel: pd.DataFrame) -> pd.DataFrame:
    df = panel.sort_values(["date", "theme_id"]).copy()
    strategies = {}

    strategies["Leading"] = df["RRG_state"].eq("Leading")
    strategies["Improving"] = df["RRG_state"].eq("Improving")
    strategies["Improving_APWC_Narrative"] = df["RRG_state"].eq("Improving") & df["APWC_high"] & df["Narrative_high"]
    strategies["Confirmed_Leading"] = df["RRG_state"].eq("Leading") & df["APWC_high"] & df["Narrative_high"]
    strategies["ExitAware_Improving_Leading"] = df["RRG_state"].isin(["Improving", "Leading"]) & df["APWC_high"] & df["Narrative_high"] & (df["RMOM_60D"] > 0)

    out = []
    for name, mask in strategies.items():
        tmp = df.copy()
        tmp["selected"] = mask.astype(int)
        daily = tmp[tmp["selected"] == 1].groupby("date")["residual_return"].mean().rename(name)
        out.append(daily)
    strat = pd.concat(out, axis=1).sort_index().fillna(0.0)
    return strat

strategy_returns = make_strategy_returns(panel)
display(strategy_returns.head())
strategy_stats = pd.DataFrame({name: ann_stats(strategy_returns[name]) for name in strategy_returns.columns}).T
display(strategy_stats)

plt.figure(figsize=(10, 5))
for col in strategy_returns.columns:
    plt.plot((1 + strategy_returns[col]).cumprod(), label=col)
plt.legend()
plt.title("RRG lifecycle strategies: cumulative residual return")
plt.ylabel("Cumulative return")
plt.show()

## 22. RRG座標ヒートマップ

RRG座標をbinに分け、各binの将来残差リターンを平均します。
RRGのどの領域が将来リターンに結びつきやすいかを視覚化します。

In [ ]:
def rrg_heatmap(panel: pd.DataFrame, horizon: int = 20, bins: int = 8):
    d = panel[["RSRatio", "RSMomentum", f"FwdRet_{horizon}D"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(d) < 100:
        print("Insufficient observations for heatmap.")
        return None
    # Winsorize coordinate range for nicer bins
    x_low, x_high = d["RSRatio"].quantile([0.02, 0.98])
    y_low, y_high = d["RSMomentum"].quantile([0.02, 0.98])
    d = d[(d["RSRatio"].between(x_low, x_high)) & (d["RSMomentum"].between(y_low, y_high))]
    d["xbin"] = pd.cut(d["RSRatio"], bins=bins)
    d["ybin"] = pd.cut(d["RSMomentum"], bins=bins)
    mat = d.pivot_table(index="ybin", columns="xbin", values=f"FwdRet_{horizon}D", aggfunc="mean")
    plt.figure(figsize=(9, 6))
    plt.imshow(mat.values, aspect="auto", origin="lower")
    plt.colorbar(label=f"Mean {horizon}D forward residual return")
    plt.title("RRG coordinate heatmap")
    plt.xlabel("RSRatio bins")
    plt.ylabel("RSMomentum bins")
    plt.xticks(range(len(mat.columns)), [str(c) for c in mat.columns], rotation=90)
    plt.yticks(range(len(mat.index)), [str(i) for i in mat.index])
    plt.tight_layout()
    plt.show()
    return mat

heatmap_mat = rrg_heatmap(panel, horizon=20, bins=8)
if heatmap_mat is not None:
    display(heatmap_mat)

## 23. Narrative betaの可視化

最後に、rolling narrative betaの内容を確認します。
テーマごとにどのnarrativeに対して感応度を持っているかを点検することで、Narrative Signalの解釈可能性を確認します。

In [ ]:
if len(beta_panel) > 0:
    latest_date = beta_panel["date"].max()
    latest_beta = beta_panel[beta_panel["date"] == latest_date].copy()
    # show top absolute betas by theme
    top_beta = latest_beta.assign(abs_beta=lambda x: x["beta"].abs()).sort_values(["theme_id", "abs_beta"], ascending=[True, False])
    top_beta = top_beta.groupby("theme_id").head(5)
    display(top_beta.head(50))

    # heatmap for a subset
    sample_themes = latest_beta["theme_id"].drop_duplicates().head(12)
    mat = latest_beta[latest_beta["theme_id"].isin(sample_themes)].pivot(index="theme_id", columns="narrative_id", values="beta")
    plt.figure(figsize=(12, max(4, 0.35 * len(mat))))
    plt.imshow(mat.fillna(0).values, aspect="auto")
    plt.colorbar(label="Narrative beta")
    plt.title(f"Theme narrative betas at {latest_date.date()}")
    plt.yticks(range(len(mat.index)), mat.index)
    plt.xticks(range(len(mat.columns)), mat.columns, rotation=90)
    plt.tight_layout()
    plt.show()
else:
    print("No beta panel available. Check BETA_WINDOW / BETA_MIN_OBS / input data length.")

## 24. 保存

必要に応じて、主要な出力をCSVに保存します。`CONFIG.SAVE_OUTPUTS=True`に変更してください。

In [ ]:
if CONFIG.SAVE_OUTPUTS:
    out_dir = Path(CONFIG.OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    panel.to_csv(out_dir / "rrg_beta_lifecycle_panel.csv", index=False)
    beta_panel.to_csv(out_dir / "theme_narrative_beta_panel.csv", index=False)
    narr_signal.to_csv(out_dir / "theme_narrative_beta_signal.csv", index=False)
    rrg_state_summary.to_csv(out_dir / "rrg_state_forward_return_summary.csv", index=False)
    rrg_apwc_summary.to_csv(out_dir / "rrg_state_apwc_forward_return_summary.csv", index=False)
    rrg_narr_summary.to_csv(out_dir / "rrg_state_narrative_forward_return_summary.csv", index=False)
    rrg_apwc_narr_summary.to_csv(out_dir / "rrg_state_apwc_narrative_forward_return_summary.csv", index=False)
    transition_summary.to_csv(out_dir / "rrg_transition_forward_return_summary.csv", index=False)
    episodes.to_csv(out_dir / "rrg_state_episodes.csv", index=False)
    strategy_returns.to_csv(out_dir / "rrg_lifecycle_strategy_returns.csv")
    print("Saved outputs to", out_dir)
else:
    print("CONFIG.SAVE_OUTPUTS is False. No files saved.")

## 25. 解釈ガイド

このノートブックで最も重視する結果は以下です。

1. **RRG状態別forward residual return**  
   `Improving`や`Leading`が将来残差リターンを識別できるか。

2. **RRG状態 × APWC**  
   APWCが高い`Improving`や`Leading`の方が、APWCが低い同状態よりも将来残差リターンが高いか。

3. **RRG状態 × Narrative Signal**  
   Narrative beta方式で作ったNarrative Signalが正の状態で、将来残差リターンが改善するか。

4. **Improving × APWC × Narrative の相互作用**  
   早期テーマ捕捉の中核仮説です。`Improving`状態、APWC高、Narrative Signal高が揃ったテーマのforward residual returnが高いかを確認します。

5. **Leading Entryの質**  
   APWC高・Narrative高を伴うLeading入りは、False Leadingが少なく、Leading入り後の残差リターンが高いか。

この5点が確認できれば、RRGを単なる可視化ではなく、Narrative betaとAPWCを組み合わせたテーマ・ライフサイクル別の残差リターン予測モデルとして利用できます。